# Height Prediction Model Error Analysis

This notebook focuses on model-level EDA for the saved height prediction checkpoint.

The goal is to understand where the model struggles on the held-out test set, especially by true height range.

## 1. Setup

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

repo_root = Path.cwd()
model_eda_dir = repo_root / "Model_EDA"

if not model_eda_dir.exists() and repo_root.name == "Model_EDA":
    repo_root = repo_root.parent
    model_eda_dir = repo_root / "Model_EDA"

sys.path.insert(0, str(model_eda_dir))

from model_error_eda import (
    COMPARISON_PATH,
    PLOTS_DIR,
    PREDICTIONS_PATH,
    PRIMARY_MODEL_NAME,
    REPORT_PATH,
    build_report,
    evaluate_models,
    make_output_dirs,
    plot_actual_vs_predicted,
    plot_error_distribution,
    plot_height_bin_errors,
    plot_model_comparison,
    plot_residuals_by_height,
    plot_worst_examples,
    save_predictions,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

make_output_dirs()
print(f"Repo root: {repo_root}")
print(f"Model EDA directory: {model_eda_dir}")

## 2. Run the Model Evaluation

This step generates predictions on the held-out test set and computes residual-based metrics.

In [ ]:
predictions_df, comparison_df = evaluate_models()
save_predictions(predictions_df, comparison_df)

primary_df = predictions_df.loc[predictions_df["model_name"] == PRIMARY_MODEL_NAME].copy()
bin_summary = plot_height_bin_errors(primary_df, PRIMARY_MODEL_NAME)
worst_examples = plot_worst_examples(primary_df, PRIMARY_MODEL_NAME)
plot_model_comparison(comparison_df)
plot_actual_vs_predicted(primary_df, PRIMARY_MODEL_NAME)
plot_error_distribution(primary_df, PRIMARY_MODEL_NAME)
plot_residuals_by_height(primary_df, PRIMARY_MODEL_NAME)
build_report(predictions_df, comparison_df, bin_summary, worst_examples)

print("Model EDA artifacts generated.")

## 3. Summary Tables

In [ ]:
comparison_df.round(2)

In [ ]:
bin_summary.round(2)

In [ ]:
worst_examples[["id", "true_height_cm", "predicted_height_cm", "error_cm", "abs_error_cm", "height_bin"]].round(2)

## 4. Written Report

In [ ]:
display(Markdown(REPORT_PATH.read_text(encoding="utf-8")))

## 5. Generated Plots

In [ ]:
plot_files = [
    "model_comparison_test_rmse.png",
    f"actual_vs_predicted_{PRIMARY_MODEL_NAME}.png",
    f"error_distribution_{PRIMARY_MODEL_NAME}.png",
    f"residuals_by_true_height_{PRIMARY_MODEL_NAME}.png",
    f"mse_by_height_bin_{PRIMARY_MODEL_NAME}.png",
    f"mae_by_height_bin_{PRIMARY_MODEL_NAME}.png",
    f"bias_by_height_bin_{PRIMARY_MODEL_NAME}.png",
]

for plot_name in plot_files:
    plot_path = PLOTS_DIR / plot_name
    display(Markdown(f"### {plot_name}"))
    display(Image(filename=str(plot_path)))